# 0) Gruplanmis Dataset Hazirlama (Colab)

Notebook 2 egitiminden once kopya farkindalikli ve taksonomi uyumlu dataset hazirlama akisidir.

Onerilen kullanim sirasi:
1. Parametreleri duzenleyin.
2. Guncelleme/erisim kontrolu hucresini calistirin.
3. Audit hucresini calistirin.
4. Cikti dosyalarini `guided/00_start_here.md` ile okuyun.
5. Her sey temizse runtime dataset materyalizasyonunu acin.


In [ ]:
from google.colab import drive, userdata

import os
import subprocess
import sys
from pathlib import Path
from urllib.parse import urlsplit, urlunsplit

drive.mount('/content/drive')

HF_TOKEN_NAMES = ("HF_TOKEN", "HUGGINGFACE_TOKEN", "HUGGINGFACE_HUB_TOKEN")
GITHUB_TOKEN_NAMES = ("GH_TOKEN", "GITHUB_TOKEN")

def _resolve_token(token_names: tuple[str, ...], canonical_env_name: str) -> str | None:
    for env_name in token_names:
        token = str(os.environ.get(env_name, "")).strip()
        if token:
            os.environ.setdefault(canonical_env_name, token)
            return token
    for secret_name in token_names:
        try:
            token = str(userdata.get(secret_name) or "").strip()
        except Exception:
            token = ""
        if token:
            os.environ[canonical_env_name] = token
            return token
    return None

hf_token = _resolve_token(HF_TOKEN_NAMES, "HF_TOKEN")
gh_token = _resolve_token(GITHUB_TOKEN_NAMES, "GH_TOKEN")
if gh_token:
    print('[GIT] GitHub token Colab secret/env uzerinden bulundu.')
else:
    print('[GIT] GitHub token bulunamadi. Public read disinda clone/push gerekiyorsa GH_TOKEN ekleyin.')

if hf_token:
    print('[HF] Hugging Face token bulundu. Gerekirse gated model indirmelerinde kullanilacak.')
else:
    print('[HF] Hugging Face token bulunamadi. Gated model gerekiyorsa Colab secret ekleyin.')

CLONE_TARGET = Path('/content/bitirmeprojesi')
COMMON_REPO_CANDIDATES = (
    Path('/content/bitirme projesi'),
    CLONE_TARGET,
    Path('/content/aads_ulora'),
    Path('/content/drive/MyDrive/bitirme projesi'),
    Path('/content/drive/MyDrive/bitirmeprojesi'),
)

REPO_URL = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/bitirmeprojesi.git')

def _is_repo_root_inline(path: Path) -> bool:
    return (path / 'src').is_dir() and (path / 'config').is_dir() and (path / 'scripts').is_dir()

def _build_repo_access_url(repo_url: str, token: str | None) -> str:
    if not token:
        return repo_url
    parsed = urlsplit(str(repo_url or '').strip())
    if parsed.scheme != 'https' or not parsed.netloc:
        return repo_url
    netloc = parsed.netloc.split('@', 1)[-1]
    return urlunsplit((parsed.scheme, f'{token}@{netloc}', parsed.path, parsed.query, parsed.fragment))

def _find_repo_root_inline() -> Path | None:
    for raw in (os.environ.get('AADS_REPO_ROOT'), os.environ.get('REPO_ROOT')):
        if not raw:
            continue
        candidate = Path(raw).expanduser().resolve()
        if _is_repo_root_inline(candidate):
            return candidate
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if _is_repo_root_inline(candidate):
            return candidate
    for candidate in COMMON_REPO_CANDIDATES:
        if _is_repo_root_inline(candidate):
            return candidate
    if CLONE_TARGET.exists() and any(CLONE_TARGET.iterdir()):
        for child in CLONE_TARGET.iterdir():
            if child.is_dir() and _is_repo_root_inline(child):
                return child
    return None

def _ensure_repo_root_for_update_check() -> Path | None:
    repo_root = _find_repo_root_inline()
    if repo_root is not None:
        return repo_root
    drive.mount('/content/drive', force_remount=False)
    repo_root = _find_repo_root_inline()
    if repo_root is not None:
        return repo_root
    clone_url = _build_repo_access_url(REPO_URL, gh_token)
    CLONE_TARGET.parent.mkdir(parents=True, exist_ok=True)
    completed = subprocess.run(
        ['git', 'clone', '--depth', '1', clone_url, str(CLONE_TARGET)],
        check=False,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    if completed.stdout:
        print(completed.stdout)
    return _find_repo_root_inline()

repo_root_for_update_check = _ensure_repo_root_for_update_check()
if repo_root_for_update_check is not None:
    if str(repo_root_for_update_check) not in sys.path:
        sys.path.insert(0, str(repo_root_for_update_check))
    try:
        from scripts.colab_repo_bootstrap import probe_repo_update_status

        update_status = probe_repo_update_status(repo_root_for_update_check)
        relation = str(update_status.get('relation', 'unknown'))
        if relation == 'up_to_date':
            print('[KONTROL] Ilk hucre: notebook/repo guncel gorunuyor.')
        elif relation == 'update_available':
            print(f"[KONTROL] Ilk hucre: guncelleme var. Branch={update_status.get('branch', '')}")
        else:
            print(f"[KONTROL] Ilk hucre: guncelleme durumu okunamadi: {update_status.get('message', 'bilgi yok')}")
    except Exception as exc:
        print(f'[KONTROL] Ilk hucrede guncellik kontrolu basarisiz: {exc}')
else:
    print('[KONTROL] Ilk hucrede repo bulunamadi ve auto-clone da basarisiz oldu.')


In [ ]:
import io
import json
import os
import random
import shutil
import sys
import subprocess
import time
from contextlib import contextmanager
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import urlsplit, urlunsplit

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch

# --- Bootstrap: repo kokunu bul, bagimliliklari kur ---
GITHUB_TOKEN_NAMES = ('GH_TOKEN', 'GITHUB_TOKEN')
CLONE_TARGET = Path('/content/bitirmeprojesi')
REPO_URL = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/bitirmeprojesi.git')
COMMON_REPO_CANDIDATES = (
    Path('/content/bitirme projesi'),
    CLONE_TARGET,
    Path('/content/aads_ulora'),
    Path('/content/drive/MyDrive/bitirme projesi'),
    Path('/content/drive/MyDrive/bitirmeprojesi'),
)

def _is_repo_root(path: Path) -> bool:
    return (path / 'src').is_dir() and (path / 'config').is_dir() and (path / 'scripts').is_dir()

def _running_in_colab_inline() -> bool:
    try:
        import google.colab  # noqa: F401
    except Exception:
        return False
    return True

def _mount_drive_inline() -> None:
    if not _running_in_colab_inline():
        return
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    except Exception as exc:
        print(f'Bootstrap sirasinda Drive mount atlandi: {exc}')

def _resolve_colab_secret_inline(secret_name: str) -> str:
    if not _running_in_colab_inline():
        return ''
    try:
        from google.colab import userdata
        return str(userdata.get(secret_name) or '').strip()
    except Exception:
        return ''

def _resolve_github_token_inline() -> str:
    for env_name in GITHUB_TOKEN_NAMES:
        token = str(os.environ.get(env_name, '')).strip()
        if token:
            os.environ.setdefault('GH_TOKEN', token)
            return token
    for secret_name in GITHUB_TOKEN_NAMES:
        token = _resolve_colab_secret_inline(secret_name)
        if token:
            os.environ['GH_TOKEN'] = token
            return token
    return ''

def _build_repo_access_url(repo_url: str, token: str) -> str:
    if not token:
        return repo_url
    parsed = urlsplit(str(repo_url or '').strip())
    if parsed.scheme != 'https' or not parsed.netloc:
        return repo_url
    netloc = parsed.netloc.split('@', 1)[-1]
    return urlunsplit((parsed.scheme, f'{token}@{netloc}', parsed.path, parsed.query, parsed.fragment))

def _find_repo_root_inline() -> Path | None:
    for raw in (os.environ.get('AADS_REPO_ROOT'), os.environ.get('REPO_ROOT')):
        if not raw:
            continue
        candidate = Path(raw).expanduser().resolve()
        if _is_repo_root(candidate):
            return candidate
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if _is_repo_root(candidate):
            return candidate
    for candidate in COMMON_REPO_CANDIDATES:
        if _is_repo_root(candidate):
            return candidate
    if CLONE_TARGET.exists() and any(CLONE_TARGET.iterdir()):
        for child in CLONE_TARGET.iterdir():
            if child.is_dir() and _is_repo_root(child):
                return child
    return None

def _ensure_repo_root_for_bootstrap() -> Path:
    repo_root = _find_repo_root_inline()
    if repo_root is not None:
        return repo_root
    _mount_drive_inline()
    repo_root = _find_repo_root_inline()
    if repo_root is not None:
        return repo_root
    clone_url = _build_repo_access_url(REPO_URL, _resolve_github_token_inline())
    CLONE_TARGET.parent.mkdir(parents=True, exist_ok=True)
    completed = subprocess.run(
        ['git', 'clone', '--depth', '1', clone_url, str(CLONE_TARGET)],
        check=False,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    if completed.stdout:
        print(completed.stdout)
    repo_root = _find_repo_root_inline()
    if repo_root is not None:
        return repo_root
    raise RuntimeError(
        'Repo bootstrap basarisiz oldu. Mevcut bir checkout icin AADS_REPO_ROOT ayarlayin veya '
        'private GitHub auto-clone icin GH_TOKEN/GITHUB_TOKEN saglayin.'
    )

BOOTSTRAP_ROOT = _ensure_repo_root_for_bootstrap()
os.chdir(BOOTSTRAP_ROOT)
if str(BOOTSTRAP_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOTSTRAP_ROOT))

from scripts.colab_repo_bootstrap import (
    export_current_colab_notebook,
    install_colab_requirements,
    login_and_check_hf_token,
    mirror_checkpoint_state_to_repo,
    mirror_path_to_repo,
    mount_drive_if_available,
    push_repo_run_to_github,
    resolve_github_token,
    resolve_hf_token,
    resolve_repo_root,
    running_in_colab,
)

IN_COLAB = running_in_colab()
if IN_COLAB:
    mount_drive_if_available(force_remount=False)

ROOT = resolve_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

install_colab_requirements(ROOT / 'colab_notebooks' / 'requirements_colab.txt', IN_COLAB)

from scripts.colab_checkpointing import TrainingCheckpointManager
from scripts.colab_live_telemetry import ColabLiveTelemetry
from src.core.config_manager import ConfigurationManager


def _install_capture_cell_output_compat() -> bool:
    if hasattr(ColabLiveTelemetry, 'capture_cell_output'):
        return False

    def _slugify_capture_name(value: str) -> str:
        slug = ''.join(ch.lower() if ch.isalnum() else '_' for ch in str(value or '').strip())
        while '__' in slug:
            slug = slug.replace('__', '_')
        slug = slug.strip('_')
        return slug or 'cell'

    class _CompatTeeTextStream:
        def __init__(self, *streams):
            self._streams = [stream for stream in streams if stream is not None]

        def write(self, data):
            text = str(data)
            for stream in self._streams:
                stream.write(text)
            return len(text)

        def flush(self):
            for stream in self._streams:
                flush = getattr(stream, 'flush', None)
                if callable(flush):
                    flush()

        def isatty(self):
            return any(bool(getattr(stream, 'isatty', lambda: False)()) for stream in self._streams)

        def writable(self):
            return True

CAPTURE_CELL_OUTPUT_COMPAT_INSTALLED = _install_capture_cell_output_compat()
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
NOTEBOOK_FILENAME = '0_grouped_dataset_preparation.executed.ipynb'
REPO_RUN_DIR = ROOT / 'runs' / RUN_ID
REPO_NOTEBOOK_OUTPUT_PATH = REPO_RUN_DIR / 'notebooks' / NOTEBOOK_FILENAME
LOCAL_OUTPUT_DIR = ROOT / 'outputs' / 'colab_notebook_data_prep'
REPO_OUTPUT_DIR = REPO_RUN_DIR / 'outputs' / 'colab_notebook_data_prep'
REPO_TELEMETRY_DIR = REPO_RUN_DIR / 'telemetry'
REPO_CHECKPOINT_STATE_DIR = REPO_RUN_DIR / 'checkpoint_state'

TELEMETRY = ColabLiveTelemetry(
    notebook_name='0_grouped_dataset_preparation.ipynb',
    run_id=RUN_ID,
)
CHECKPOINT_ROOT = TELEMETRY.drive_run_dir if TELEMETRY.drive_run_dir.exists() else TELEMETRY.local_run_dir
CHECKPOINT_MANAGER = TrainingCheckpointManager(CHECKPOINT_ROOT, retention=3)
DEVICE = str(ConfigurationManager(config_dir=str(ROOT / 'config'), environment='colab').load_all_configs().get('training', {}).get('continual', {}).get('device', 'cuda'))

LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPO_NOTEBOOK_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

def rt(message: str, *, phase: str = 'notebook') -> None:
    text = str(message)
    print(text)
    TELEMETRY.emit_log(text, phase=phase, level='info')

def save_run_outputs_to_repo() -> dict[str, str]:
    exports: dict[str, str] = {}

    mirrored_outputs = mirror_path_to_repo(LOCAL_OUTPUT_DIR, REPO_OUTPUT_DIR)
    if mirrored_outputs is not None:
        exports['outputs'] = str(mirrored_outputs)

    telemetry_source = TELEMETRY.drive_run_dir if TELEMETRY.drive_run_dir.exists() else TELEMETRY.local_run_dir
    mirrored_telemetry = mirror_path_to_repo(telemetry_source, REPO_TELEMETRY_DIR)
    if mirrored_telemetry is not None:
        exports['telemetry'] = str(mirrored_telemetry)

    mirrored_checkpoint_state = mirror_checkpoint_state_to_repo(CHECKPOINT_ROOT, REPO_CHECKPOINT_STATE_DIR)
    if mirrored_checkpoint_state is not None:
        exports['checkpoint_state'] = str(mirrored_checkpoint_state)

    return exports

def _copy_path_to_drive_exports(source_path: Path, relative_name: str) -> Path | None:
    source = Path(source_path).expanduser()
    if not source.exists():
        return None
    drive_root = TELEMETRY.drive_run_dir if TELEMETRY.drive_run_dir.exists() else None
    if drive_root is None:
        return None
    target = drive_root / 'artifacts' / relative_name
    if source.is_dir():
        if target.exists():
            shutil.rmtree(target)
        shutil.copytree(source, target)
    else:
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, target)
    return target

TELEMETRY.configure_repo_output_export(
    output_dir=REPO_OUTPUT_DIR,
    notebook_filename=NOTEBOOK_FILENAME,
    export_notebook_fn=export_current_colab_notebook,
)
TELEMETRY.update_latest({'phase': 'bootstrap_ready', 'run_id': RUN_ID})
print(f'[BOOTSTRAP] run_id={RUN_ID} capture_cell_output_compat={CAPTURE_CELL_OUTPUT_COMPAT_INSTALLED}')


In [ ]:
with TELEMETRY.capture_cell_output("Cell 3: Parameters"):
    # --- Notebook 0 parametreleri ---
    # Once bu hucreyi duzenleyin, sonra alttaki hucreleri sirayla calistirin.

    # DATASET_ROOT: desteklenen sinif klasorlerini iceren duz class-root dataset.
    DATASET_ROOT = "data/class_root_dataset"

    # CROP_NAME: taxonomy hizalamasi icin crop anahtari.
    CROP_NAME = "tomato"

    # PART_NAME: hazirlanan runtime dataset klasorunu crop bazinda ayristirmak icin kullanilir.
    PART_NAME = "unspecified"

    # PROVENANCE_MANIFEST_PATH: opsiyonel provenance CSV sidecar yolu; bos ise <class_root>/provenance_manifest.csv otomatik aranir.
    PROVENANCE_MANIFEST_PATH = ""


    # PREP_ARTIFACT_ROOT: audit manifestleri ve review raporlarinin yazilacagi yer.
    PREP_ARTIFACT_ROOT = "outputs/colab_notebook_data_prep/artifacts"

    # PREPARED_RUNTIME_ROOT: onaydan sonra runtime dataset'in materyalize edilecegi kok.
    PREPARED_RUNTIME_ROOT = "data/prepared_runtime_datasets"

    # PREPARED_CLASS_ROOT: audit raporlarindan uretilen temiz class-root kopyasinin yazilacagi kok.
    PREPARED_CLASS_ROOT = "data/prepared_class_root_datasets"

    # PREPARE_DATASET_FROM_REPORTS: exact duplicate ve guvenli review sonucunu kullanarak materyalizasyon oncesi calisma kopyasi hazirlar.
    PREPARE_DATASET_FROM_REPORTS = True

    # CONFIRM_PREPARE_FOR_MATERIALIZATION: Cell 4b'nin rapor ozetini inceledikten sonra 'prepare' yapin.
    CONFIRM_PREPARE_FOR_MATERIALIZATION = ""

    # MATERIALIZE_AFTER_REVIEW: sadece audit/prep icin False birakin; temiz ise True yapin.
    MATERIALIZE_AFTER_REVIEW = False

    # CLEANUP_SEED: rapor-tabanli temizleme secimlerini deterministik tutar.
    CLEANUP_SEED = 42

    # DEVICE: DINOv3 ve BioCLIP embedding cikarimi icin cihaz.
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    # EMBEDDING_BATCH_SIZE: embedding batch boyutu.
    EMBEDDING_BATCH_SIZE = 32 if DEVICE == "cuda" else 8

    # NEIGHBORS: ayni sinif aday ciftleri icin komsu sayisi.
    NEIGHBORS = 4

    PREP_DINOV3_MODEL_ID = "facebook/dinov3-vitl16-pretrain-lvd1689m"
    PREP_BIOCLIP_MODEL_ID = "imageomics/bioclip-2.5-vith14"

    run_id = RUN_ID
    STATE = {
        "validated": False,
        "audit_summary": None,
        "artifact_root": None,
        "runtime_dataset_root": None,
        "dataset_root": None,
        "provenance_manifest_path": None,
        "prep_materialization_result": None,
        "access_report": None,
    }
    print(f"[PARAMS] run_id={run_id} crop={CROP_NAME} part={PART_NAME} dataset_root={DATASET_ROOT}")


In [ ]:
with TELEMETRY.capture_cell_output("Cell 3b: Guncelleme ve Erisim Kontrolu"):
    from scripts.colab_repo_bootstrap import collect_notebook_access_report, print_notebook_access_report

    access_report = collect_notebook_access_report(
        repo_root=ROOT,
        hf_model_ids=[PREP_DINOV3_MODEL_ID, PREP_BIOCLIP_MODEL_ID],
    )
    STATE["access_report"] = access_report
    print_notebook_access_report(access_report, print_fn=print)
    TELEMETRY.update_latest(
        {
            "phase": "access_checked",
            "github_read_access": access_report.get("github", {}).get("read_access_mode"),
            "repo_update_relation": access_report.get("repo_updates", {}).get("relation"),
            "hf_access_mode": access_report.get("huggingface", {}).get("access_mode"),
        }
    )


In [ ]:
with TELEMETRY.capture_cell_output("Cell 4: Dataset Audit"):
    from scripts.prepare_grouped_runtime_dataset import build_grouped_dataset_plan

    dataset_root = Path(DATASET_ROOT).expanduser()
    if not dataset_root.is_absolute():
        dataset_root = (ROOT / dataset_root).resolve()
    if not dataset_root.is_dir():
        raise RuntimeError(f"Dataset root bulunamadi: {dataset_root}")

    artifact_root = Path(PREP_ARTIFACT_ROOT).expanduser()
    if not artifact_root.is_absolute():
        artifact_root = (ROOT / artifact_root).resolve()

    provenance_manifest_path = str(PROVENANCE_MANIFEST_PATH).strip()
     if provenance_manifest_path:
         provenance_manifest_path = Path(provenance_manifest_path).expanduser()
         if not provenance_manifest_path.is_absolute():
             provenance_manifest_path = (ROOT / provenance_manifest_path).resolve()
     else:
         provenance_manifest_path = None
 
    summary = build_grouped_dataset_plan(
        class_root=dataset_root,
        crop_name=CROP_NAME,
        artifact_root=artifact_root,
        provenance_manifest_path=provenance_manifest_path,
        taxonomy_path=ROOT / "config" / "plant_taxonomy.json",
        dino_model_id=PREP_DINOV3_MODEL_ID,
        bioclip_model_id=PREP_BIOCLIP_MODEL_ID,
        device=DEVICE,
        batch_size=EMBEDDING_BATCH_SIZE,
        neighbors=NEIGHBORS,
        progress_fn=lambda message: print(f"[PREP] {message}"),
    )

    STATE["validated"] = True
    STATE["provenance_manifest_path"] = provenance_manifest_path
    STATE["audit_summary"] = summary
    STATE["artifact_root"] = artifact_root
    STATE["dataset_root"] = dataset_root
    print(json.dumps(summary.get("summary", {}), indent=2))
    print(f"[PREP] runtime_ready={summary.get('runtime_ready')} artifact_root={artifact_root}")
    if summary.get("blocking_issues"):
        print("[PREP] Bloklayici sorunlar:")
        for item in summary["blocking_issues"]:
            print(f"  - {item}")
    drive_artifact_root = _copy_path_to_drive_exports(artifact_root, 'data_prep_artifacts')
    if drive_artifact_root is not None:
        print(f"[PREP] Drive kopyasi guncellendi: {drive_artifact_root}")

    TELEMETRY.update_latest(
        {
            "phase": "data_prep_audited",
            "dataset_root": str(dataset_root),
            "artifact_root": str(artifact_root),
            "drive_artifact_root": ("" if drive_artifact_root is None else str(drive_artifact_root)),
            "runtime_ready": bool(summary.get("runtime_ready")),
        }
    )


In [ ]:
with TELEMETRY.capture_cell_output("Cell 4b: Prepare Dataset For Materialization"):
    from scripts.prepare_grouped_runtime_dataset import build_prepared_dataset_key
    from scripts.prepare_materialization_dataset import prepare_class_root_for_materialization

    if not STATE.get("validated") or STATE.get("audit_summary") is None:
        raise RuntimeError("Once dataset audit hucresini calistirin.")

    summary = STATE["audit_summary"]
    if not PREPARE_DATASET_FROM_REPORTS:
        print("[PREP] PREPARE_DATASET_FROM_REPORTS=False. Audit raporlari pasif birakildi.")
    else:
        dataset_key = build_prepared_dataset_key(CROP_NAME, PART_NAME)
        prepared_class_root_parent = Path(PREPARED_CLASS_ROOT).expanduser()
        if not prepared_class_root_parent.is_absolute():
            prepared_class_root_parent = (ROOT / prepared_class_root_parent).resolve()
        prepared_class_root = prepared_class_root_parent / dataset_key
        prepared_artifact_root = STATE["artifact_root"].parent / f"{STATE['artifact_root'].name}_prepared"
        prep_counts = dict(summary.get("summary", {}))
        print("[PREP][CONFIRM] Rapor tabanli hazirlik baslamadan once ozet:")
        print(
            f"  dataset_key={dataset_key} total_images={prep_counts.get('total_images', 0)} "
            f"cross_class_conflicts={prep_counts.get('cross_class_conflicts', 0)} "
            f"same_class_high_risk_clusters={prep_counts.get('same_class_high_risk_clusters', 0)}"
        )
        print(f"  source_dataset_root={STATE['dataset_root']}")
        print(f"  source_artifact_root={STATE['artifact_root']}")
        print(f"  prepared_class_root={prepared_class_root}")
        print(f"  prepared_artifact_root={prepared_artifact_root}")
        print(f"  cleanup_seed={CLEANUP_SEED}")
        confirmation_value = str(CONFIRM_PREPARE_FOR_MATERIALIZATION).strip().lower()
        if confirmation_value != "prepare":
            TELEMETRY.update_latest(
                {
                    "phase": "data_prep_prepare_confirmation_required",
                    "dataset_root": str(STATE["dataset_root"]),
                    "artifact_root": str(STATE["artifact_root"]),
                    "runtime_ready": bool(summary.get("runtime_ready")),
                    "prepared_dataset_key": dataset_key,
                }
            )
            raise RuntimeError(
                "Rapor tabanli hazirlik henuz baslatilmadi. Ozet yukarida yazdirildi. "
                "Devam etmek icin parametre hucresinde CONFIRM_PREPARE_FOR_MATERIALIZATION='prepare' yapip bu hucreyi tekrar calistirin."
            )
        prep_result = prepare_class_root_for_materialization(
            class_root=STATE["dataset_root"],
            crop_name=CROP_NAME,
            part_name=PART_NAME,
            audit_artifact_root=STATE["artifact_root"],
            provenance_manifest_path=STATE.get("provenance_manifest_path"),
            prepared_class_root=prepared_class_root,
            prepared_artifact_root=prepared_artifact_root,
            taxonomy_path=ROOT / "config" / "plant_taxonomy.json",
            dino_model_id=PREP_DINOV3_MODEL_ID,
            bioclip_model_id=PREP_BIOCLIP_MODEL_ID,
            device=DEVICE,
            batch_size=EMBEDDING_BATCH_SIZE,
            neighbors=NEIGHBORS,
            cleanup_seed=CLEANUP_SEED,
            quarantine_cross_class_conflicts=True,
            materialization_strategy="auto",
            progress_fn=lambda message: print(f"[PREP] {message}"),
        )
        STATE["prep_materialization_result"] = prep_result
        STATE["dataset_root"] = Path(prep_result["prepared_class_root"])
        STATE["artifact_root"] = Path(prep_result["prepared_artifact_root"])
        STATE["audit_summary"] = prep_result["rerun_summary"]
        summary = STATE["audit_summary"]
        print(
            f"[PREP] prepared_runtime_ready={prep_result.get('prepared_runtime_ready')} "
            f"dataset_key={prep_result.get('dataset_key')} prepared_class_root={STATE['dataset_root']}"
        )
        print(f"[PREP] Hazirlik sonrasi artifact_root={STATE['artifact_root']}")
        drive_prepared_artifact_root = _copy_path_to_drive_exports(STATE["artifact_root"], 'data_prep_artifacts_prepared')
        if drive_prepared_artifact_root is not None:
            print(f"[PREP] Drive hazirlik artifact kopyasi guncellendi: {drive_prepared_artifact_root}")
        TELEMETRY.update_latest(
            {
                "phase": "data_prep_prepared",
                "dataset_root": str(STATE["dataset_root"]),
                "artifact_root": str(STATE["artifact_root"]),
                "drive_artifact_root": ("" if drive_prepared_artifact_root is None else str(drive_prepared_artifact_root)),
                "runtime_ready": bool(summary.get("runtime_ready")),
            }
        )


In [ ]:
with TELEMETRY.capture_cell_output("Cell 5: Materialize Runtime Dataset"):
    from scripts.prepare_grouped_runtime_dataset import build_prepared_dataset_key, materialize_grouped_runtime_dataset

    if not STATE.get("validated") or STATE.get("audit_summary") is None:
        raise RuntimeError("Once dataset audit hucresini calistirin.")

    summary = STATE["audit_summary"]
    if not MATERIALIZE_AFTER_REVIEW:
        print("[PREP] MATERIALIZE_AFTER_REVIEW=False. Audit dosyalari incelemeye hazir.")
    else:
        if not summary.get("runtime_ready"):
            raise RuntimeError("Audit sonucu bloklayici sorunlar iceriyor. Materyalizasyon once temizlenmeli.")
        runtime_root = Path(PREPARED_RUNTIME_ROOT).expanduser()
        if not runtime_root.is_absolute():
            runtime_root = (ROOT / runtime_root).resolve()
        dataset_key = build_prepared_dataset_key(CROP_NAME, PART_NAME)
        runtime_dataset_root = materialize_grouped_runtime_dataset(
            class_root=STATE["dataset_root"],
            crop_name=CROP_NAME,
            part_name=PART_NAME,
            artifact_root=STATE["artifact_root"],
            runtime_root=runtime_root,
            materialization_strategy="auto",
        )
        STATE["runtime_dataset_root"] = runtime_dataset_root
        print(f"[PREP] Hazir runtime dataset su klasore yazildi: {runtime_dataset_root / dataset_key}")
        drive_runtime_root = _copy_path_to_drive_exports(runtime_dataset_root / dataset_key, f'prepared_runtime_datasets/{dataset_key}')
        if drive_runtime_root is not None:
            print(f"[PREP] Drive runtime kopyasi guncellendi: {drive_runtime_root}")
        TELEMETRY.update_latest(
            {
                "phase": "data_prep_materialized",
                "runtime_dataset_root": str(runtime_dataset_root),
                "drive_runtime_root": ("" if drive_runtime_root is None else str(drive_runtime_root)),
            }
        )

notebook_export_result = export_current_colab_notebook(REPO_NOTEBOOK_OUTPUT_PATH)
TELEMETRY.merge_summary_metadata(
    {
        "access_check": STATE.get("access_report", {}),
        "prep_summary": STATE.get("audit_summary", {}),
        "materialization_prep_summary": STATE.get("prep_materialization_result", {}),
        "repo_run_dir": str(REPO_RUN_DIR),
        "notebook_export_path": str(notebook_export_result or REPO_NOTEBOOK_OUTPUT_PATH),
    }
)
TELEMETRY.close(
    {
        "status": "ok",
        "runtime_ready": bool((STATE.get("audit_summary") or {}).get("runtime_ready")),
        "materialized": bool(STATE.get("runtime_dataset_root")),
        "repo_run_dir": str(REPO_RUN_DIR),
    }
)
REPO_RUN_EXPORTS = save_run_outputs_to_repo()
for key in sorted(REPO_RUN_EXPORTS):
    print(f"[RUNS] {key} -> {REPO_RUN_EXPORTS[key]}")
print(f"[RUNS] notebook -> {REPO_NOTEBOOK_OUTPUT_PATH}")
